# 📝 과제 01: 온라인 쇼핑 상담 4비트 불러오기와 메모리 최적화

## 만들 결과

교안 01(4비트 양자화와 GPU 메모리 최적화)의 과제입니다. 지난 과제의 온라인 쇼핑 상담(전자상거래법) 모델을 4비트로 불러오고, 학습 갱신 한 번의 메모리와 시간을 측정해 메모리 예산 안에서 학습 설정을 고릅니다. 학습은 과제 02에서 합니다.

- 1~3번: 4비트 설정 `bnb_config`, 모델 메모리 테이블 `footprint_table`, 어댑터를 붙인 `qlora_model`
- 4~6번: 측정 테이블 `load_table`, `checkpointing_table`, `commerce_table`, 고른 설정 `chosen`과 학습 설정 `chosen_args`

## 준비

- **자료:** `data/과제/commerce_train.jsonl`(지난 과제와 같은 학습 데이터 135행)
- **제공 코드:** 경로(`data_dir`, `output_dir`)와 `device`, `MODEL_NAME`, `tokenizer`, `load_jsonl`, `train_rows`, 학습 데이터를 한 줄로 이어 붙인 토큰 열 `commerce_token_ids`, 가장 긴 예시의 토큰 수 `commerce_max_length`. 3번 뒤에 측정 도우미 `load_lora_model`, `measure_train_step`이 있습니다.
- **실행 환경:** 교안과 같은 RunPod CUDA GPU에서 실행합니다. API 키는 필요 없고, 측정값은 GPU와 실행마다 조금씩 다릅니다.
- **작성:** `[작성]` 셀의 `...`를 `# TODO` 주석대로 채우고 바로 아래 `[자가채점]` 셀을 실행합니다. 뒤 문항이 앞 문항의 변수를 쓰므로 1번부터 차례로 풉니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# 토크나이저: 학습 전후 모델이 같은 토크나이저를 씁니다
# HF_TOKEN 이 없다는 Warning 은 로그인 없이 내려받는다는 안내이며 오류가 아닙니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("대화 끝 토큰:", tokenizer.eos_token, "| 패딩 토큰:", tokenizer.pad_token)

In [ ]:
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
# 지난 과제와 같은 쇼핑 상담 학습 데이터
train_rows = load_jsonl(data_dir / "과제" / "commerce_train.jsonl")

# 예시마다 학습 때와 같은 chat template 문자열의 토큰 수를 세고, 토큰 번호를 한 줄로 이어 붙입니다
commerce_lengths = []
commerce_ids = []
for row in train_rows:
    chat_text = tokenizer.apply_chat_template(
        row["messages"],  # system, user, assistant 메시지
        tokenize=False,   # 토큰 번호가 아니라 문자열로 받습니다
    )
    row_ids = tokenizer(chat_text)["input_ids"]
    commerce_lengths.append(len(row_ids))
    commerce_ids.extend(row_ids)

# 가장 긴 예시의 토큰 수와, 측정 배치를 앞에서부터 자를 1차원 정수 텐서
commerce_max_length = max(commerce_lengths)
commerce_token_ids = torch.tensor(commerce_ids)
print("학습 행:", len(train_rows), "| 가장 긴 예시:", commerce_max_length, "토큰 | 이어 붙인 토큰 수:", len(commerce_token_ids))

## 1. 4비트 설정: BitsAndBytesConfig

**목적:** 쇼핑 상담 모델을 GPU 메모리를 적게 쓰는 4비트로 불러올 설정을 만듭니다.

**요구사항**
- (1) `BitsAndBytesConfig`로 4비트 설정 **`bnb_config`** 를 만듭니다(교안 01의 4비트 설정과 같은 값).
- (2) `load_in_4bit`를 `True`로 둡니다.
- (3) `bnb_4bit_quant_type`을 기본값 `"fp4"` 대신 `"nf4"`로 둡니다.
- (4) `bnb_4bit_use_double_quant`를 `True`로 둡니다.
- (5) `bnb_4bit_compute_dtype`을 기본값 float32 대신 `torch.bfloat16`으로 둡니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 4-1 4비트 설정 셀과 같은 방법입니다.
```

</details>

In [ ]:
# [작성] 4비트 설정을 만드세요.
from transformers import BitsAndBytesConfig

# TODO (1) BitsAndBytesConfig 에 아래 (2)~(5) 인자를 한 줄에 하나씩 넣습니다
bnb_config = BitsAndBytesConfig(
    # TODO (2) 선형 층을 4비트로 불러오는 인자를 켭니다
    # TODO (3) 양자화 형식을 NF4 로 고릅니다
    # TODO (4) 이중 양자화를 켭니다
    # TODO (5) 4비트 층의 계산 자료형을 torch 의 bf16 으로 둡니다
    ...
)

In [ ]:
# [자가채점] 4비트 설정의 네 값을 확인합니다
assert isinstance(bnb_config, BitsAndBytesConfig), "bnb_config는 transformers의 BitsAndBytesConfig입니다. `...`를 채웠는지 확인하세요."
assert bnb_config.load_in_4bit is True, "load_in_4bit=True를 넣으세요."
assert bnb_config.bnb_4bit_quant_type == "nf4", f"bnb_4bit_quant_type이 {bnb_config.bnb_4bit_quant_type}입니다. 기본값 fp4 대신 NF4를 고르세요."
assert bnb_config.bnb_4bit_use_double_quant is True, "bnb_4bit_use_double_quant=True를 넣으세요(블록마다 둔 absmax도 한 번 더 양자화합니다)."
assert bnb_config.bnb_4bit_compute_dtype == torch.bfloat16, f"bnb_4bit_compute_dtype이 {bnb_config.bnb_4bit_compute_dtype}입니다. 기본값 float32 대신 torch의 bf16 자료형을 넣으세요."
print("✅ 통과!")

## 2. 4비트로 불러오기와 모델 메모리: from_pretrained, get_memory_footprint

**목적:** 작은 GPU에 올릴 상담 모델을 4비트로 불러 float32 Base보다 모델 메모리가 얼마나 줄었는지 기록합니다.

**요구사항**
- (1) 지난 과제와 같이 `MODEL_NAME`을 `torch.float32`로 불러 `device`에 올린 **`base_model`**(float32 Base)을 만듭니다.
- (2) `bnb_config`를 `quantization_config`로 넘겨 `MODEL_NAME`을 불러온 **`base_4bit`** 를 만듭니다. 4비트로 바뀌지 않는 층은 `torch.bfloat16`, 자리는 GPU 0번으로 정하고 `.to(device)`는 쓰지 않습니다.
- (3) 열이 `"모델"`, `"모델 메모리(GB)"`(이 순서)인 `pd.DataFrame` **`footprint_table`** 을 만듭니다. `"모델"` 열의 값은 `"float32 Base"`, `"4비트 Base"` 순서입니다.
- (4) `"모델 메모리(GB)"`는 각 모델의 `get_memory_footprint()`(바이트)를 `1024 ** 3`으로 나눠 소수 둘째 자리로 반올림한 값이고, 4비트 Base는 3번에서 어댑터를 붙이기 전(불러온 직후)에 계산합니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 4-2 4비트로 불러오기 셀, 4-3 가중치 메모리 계산과 측정 셀과 같은 방법입니다.
float32 Base 는 지난 과제와 같은 방법으로 불러옵니다.
```

</details>

In [ ]:
# [작성] float32 Base 와 4비트 Base 를 불러오고 모델 메모리 테이블을 만드세요.

# TODO (1) AutoModelForCausalLM.from_pretrained 로 float32 Base 를 불러와 device 에 올립니다
base_model = ...

# TODO (2) from_pretrained 에 4비트 설정, 남는 층의 bf16 dtype, GPU 0번 device_map 을 넘깁니다(.to(device) 는 쓰지 않습니다)
base_4bit = ...

# TODO (3) "모델", "모델 메모리(GB)" 두 열과 float32 Base, 4비트 Base 두 행으로 pd.DataFrame 을 만듭니다
# TODO (4) 모델 메모리는 get_memory_footprint() 를 1024 ** 3 으로 나눠 round 로 소수 둘째 자리까지 남깁니다
footprint_table = ...
display(footprint_table)

In [ ]:
# [자가채점] 두 모델의 자료형과 4비트 층, 모델 메모리 테이블을 확인합니다
assert next(base_model.parameters()).dtype == torch.float32, "base_model은 MODEL_NAME을 float32로 불러온 모델입니다."
gate_type = type(base_4bit.model.layers[0].mlp.gate_proj).__name__
assert gate_type == "Linear4bit", f"base_4bit의 첫 층 gate_proj가 {gate_type}입니다. quantization_config로 bnb_config를 넘겼는지 확인하세요."
assert isinstance(footprint_table, pd.DataFrame) and list(footprint_table.columns) == ["모델", "모델 메모리(GB)"], f"footprint_table의 열이 {list(footprint_table.columns)}입니다. \"모델\", \"모델 메모리(GB)\" 순서입니다."
assert list(footprint_table["모델"]) == ["float32 Base", "4비트 Base"], f"footprint_table의 모델 열이 {list(footprint_table['모델'])}입니다. float32 Base, 4비트 Base 순서입니다."
float32_gb = footprint_table["모델 메모리(GB)"].iloc[0]
four_bit_gb = footprint_table["모델 메모리(GB)"].iloc[1]
assert float32_gb > 1, f"float32 Base 모델 메모리가 {float32_gb}GB입니다. get_memory_footprint()를 1024 ** 3 으로 나눴는지 확인하세요."
assert float32_gb / 8 < four_bit_gb < float32_gb / 3, f"4비트 Base 모델 메모리 {four_bit_gb}GB가 float32 Base {float32_gb}GB의 1/8~1/3 범위 밖입니다. 2번에서 quantization_config로 bnb_config를 넘기고, 3번에서 어댑터를 붙이기 전(불러온 직후)의 base_4bit로 계산했는지 확인하세요."
print("✅ 통과!")

## 3. LoRA 어댑터 붙이기: get_peft_model, get_nb_trainable_parameters

**목적:** 4비트로 불러온 모델에 지난 과제와 같은 LoRA 어댑터를 붙이고, `SFTTrainer`처럼 어댑터를 bf16으로 맞춰 학습할 수 있게 합니다.

**요구사항**
- (1) `get_peft_model`로 `base_4bit`에 셀에 적어 둔 `lora_config`(바꾸지 않습니다)를 붙인 **`qlora_model`**(`PeftModel`)을 만듭니다.
- (2) `qlora_model.parameters()`를 돌며 학습하는 파라미터(`requires_grad`가 `True`)만 `.data`를 `torch.bfloat16`으로 바꿉니다. 4비트 모델을 받은 `SFTTrainer`가 어댑터에 하는 일과 같고, 임베딩과 정규화 층은 2번에서 불러온 bf16 그대로 둡니다.
- (3) `qlora_model.get_nb_trainable_parameters()`의 두 반환값을 차례로 **`trainable_count`**(학습 파라미터 수), **`all_count`**(전체 파라미터 수)에 담습니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 5-2 LoRA 어댑터 붙이기 셀과 같은 방법입니다.
SFTTrainer 가 4비트 모델의 어댑터를 bf16 으로 바꾸는 이유는 교안 01 5-1 에 있습니다.
get_nb_trainable_parameters 는 지난 시간 교안 01 4-1 학습 파라미터 수 셀에서 썼습니다.
```

</details>

In [ ]:
# [작성] 4비트 Base 에 지난 과제와 같은 어댑터를 붙이고, 학습할 어댑터를 bf16 으로 바꾸세요.
from peft import get_peft_model

# 지난 과제와 같은 LoRA 설정(그대로 씁니다)
from peft import LoraConfig

# 지난 시간 본실습과 같은 설정: 출력층을 뺀 모든 선형 층에 r=8 어댑터, 배율 16 / 8 = 2.0
lora_config = LoraConfig(
    r=8,                          # B, A 사이의 크기(rank)
    lora_alpha=16,                # 배율 = lora_alpha / r = 2.0
    target_modules="all-linear",  # lm_head를 뺀 모든 선형 층
    lora_dropout=0.05,            # 어댑터 입력에 거는 드롭아웃 비율
    task_type="CAUSAL_LM",        # 다음 토큰을 예측하는 생성 모델
)

# TODO (1) get_peft_model 로 base_4bit 에 lora_config 어댑터를 붙입니다
qlora_model = ...

# TODO (2) qlora_model.parameters() 를 돌며 requires_grad 가 True 인 파라미터만 .data 를 torch.bfloat16 으로 바꿉니다
...

# TODO (3) qlora_model 의 get_nb_trainable_parameters() 가 돌려준 두 값을 담습니다
trainable_count, all_count = ...
qlora_model.print_trainable_parameters()

In [ ]:
# [자가채점] LoRA 설정, 학습 파라미터 수, 고정한 4비트 베이스 층, 임베딩과 어댑터의 자료형을 확인합니다
from peft import PeftModel

assert isinstance(qlora_model, PeftModel), "qlora_model은 base_4bit에 lora_config로 어댑터를 붙인 PeftModel입니다."
assert lora_config.r == 8 and lora_config.lora_alpha == 16, "셀에 적어 둔 lora_config를 바꾸지 말고 그대로 쓰세요."
assert (trainable_count, all_count) == qlora_model.get_nb_trainable_parameters(), "trainable_count, all_count 는 get_nb_trainable_parameters()의 두 반환값입니다."
assert trainable_count == 5411328, f"학습 파라미터 수가 {trainable_count:,}입니다(지난 과제 LoRA 5,411,328). 셀에 적어 둔 lora_config로 어댑터를 붙였는지 확인하세요."
qlora_gate = qlora_model.base_model.model.model.layers[0].mlp.gate_proj
assert type(qlora_gate.base_layer).__name__ == "Linear4bit", "어댑터를 붙인 gate_proj의 베이스 층이 4비트 층이 아닙니다. 2번의 4비트 Base에 어댑터를 붙였는지 확인하세요."
assert qlora_gate.base_layer.weight.requires_grad is False, "4비트 베이스 가중치가 학습 상태(requires_grad=True)입니다. get_peft_model 이 베이스를 고정하므로, 직접 requires_grad 를 바꾸지 말고 2번부터 다시 실행하세요."
embedding_dtype = qlora_model.get_input_embeddings().weight.dtype
assert embedding_dtype == torch.bfloat16, f"입력 임베딩이 {embedding_dtype}입니다(기대값 bfloat16). 2번에서 4비트로 바뀌지 않는 층의 dtype을 torch.bfloat16으로 넘겼는지 확인하고, 2번부터 다시 실행하세요."
adapter_dtype = qlora_gate.lora_A["default"].weight.dtype
assert adapter_dtype == torch.bfloat16, f"어댑터 A의 자료형이 {adapter_dtype}입니다(기대값 bfloat16). (2)에서 requires_grad 가 True 인 파라미터의 .data 를 torch.bfloat16 으로 바꿨는지 확인하세요."
print("✅ 통과!")

### 학습 메모리 측정 준비

아래 셀들은 교안 01과 같은 측정 도우미라 실행만 하고 넘어갑니다. `load_lora_model(model_name, load_mode, gradient_checkpointing)`은 `load_mode`(`"float32"`, `"bf16"`, `"4bit"`)로 불러와 `lora_config` 어댑터를 붙인 모델을, `measure_train_step(model, optimizer, token_ids, batch_size, seq_len, accumulation_steps=1)`은 준비 갱신 한 번 뒤 측정 구간의 결과를 딕셔너리(키 `"배치 크기"`, `"시퀀스 길이"`, `"누적 횟수"`, `"유효 배치"`, `"최대 할당(GB)"`, `"예약(GB)"`, `"갱신당 시간(초)"`)로 돌려줍니다. 마지막 셀은 앞에서 불러온 모델을 지워 측정에 섞이지 않게 합니다.

In [ ]:
from peft import get_peft_model
from transformers import AutoModelForCausalLM


def load_lora_model(model_name, load_mode, gradient_checkpointing):
    """베이스 모델을 load_mode("float32", "bf16", "4bit")로 불러와 lora_config 어댑터를 붙인 모델을 돌려줍니다(SFTTrainer 에 peft_config 를 넘긴 것과 같게)."""
    if load_mode == "4bit":
        # QLoRA: 선형 층을 4비트로 불러옵니다. 4비트로 바뀌지 않는 층(임베딩, 정규화)은 bf16 그대로 둡니다
        model = AutoModelForCausalLM.from_pretrained(
            model_name,                      # 허브의 모델 이름
            quantization_config=bnb_config,  # 선형 층을 NF4 4비트로 불러오는 설정
            dtype=torch.bfloat16,            # 4비트로 바뀌지 않는 층(임베딩, 정규화)의 dtype
            device_map={"": 0},              # 모델 전체("")를 GPU 0번(첫 번째 GPU)에 둡니다
        )
    else:
        # LoRA: float32(지난 시간과 같음) 또는 bf16으로 불러옵니다
        if load_mode == "bf16":
            dtype = torch.bfloat16
        else:
            dtype = torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            model_name,          # 허브의 모델 이름
            dtype=dtype,         # 가중치를 둘 dtype(float32 또는 bf16)
            device_map={"": 0},  # 모델 전체("")를 GPU 0번(첫 번째 GPU)에 둡니다
        )
    if gradient_checkpointing:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},  # PyTorch 권장 체크포인팅 방식
        )

    # 세 방식 모두 같은 lora_config 로 어댑터를 붙입니다(베이스 가중치는 고정됩니다)
    model = get_peft_model(model, lora_config)
    if gradient_checkpointing:
        model.enable_input_require_grads()  # 체크포인팅을 켜면 SFTTrainer 처럼 입력 쪽 기울기 길을 열어 둡니다

    # 4비트 모델은 SFTTrainer 처럼 학습할 어댑터를 bf16으로 바꿉니다(QLoRA 논문 권고)
    if load_mode == "4bit":
        for param in model.parameters():
            if param.requires_grad:
                param.data = param.data.to(torch.bfloat16)
    return model

In [ ]:
import gc
import time


def measure_train_step(model, optimizer, token_ids, batch_size, seq_len, accumulation_steps=1, steps=3):
    """준비 갱신 1번 뒤 steps번 갱신하며 최대 할당 메모리, 예약 메모리(GB), 갱신당 시간(초)을 딕셔너리로 돌려줍니다."""
    # 앞 측정이 잡아 둔 캐시를 돌려줘, 예약 메모리를 이 설정만으로 측정합니다
    gc.collect()
    torch.cuda.empty_cache()

    # 이어 붙인 토큰 열 앞에서부터 batch_size * seq_len개를 잘라 (배치, 길이) 모양으로 만듭니다
    batch = token_ids[:batch_size * seq_len].reshape(batch_size, seq_len).to(device)

    # 첫 회(step 0)는 옵티마이저 상태를 만드는 준비 갱신이고, 나머지 steps번이 측정 구간입니다
    model.train()
    for step in range(steps + 1):
        optimizer.zero_grad()

        # 같은 배치를 accumulation_steps번 돌려 기울기를 .grad에 더합니다(메모리와 시간만 보므로 배치 내용은 같아도 됩니다)
        for _ in range(accumulation_steps):
            # 혼합 정밀도: SFTTrainer가 기본으로 쓰는 것과 같은 계산(autocast)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss = model(
                    input_ids=batch,   # 입력 토큰
                    labels=batch,      # 다음 토큰 예측의 정답(모델이 한 칸 밀어 손실을 계산합니다)
                    use_cache=False,   # 생성용 캐시를 만들지 않습니다(학습과 같은 조건)
                ).loss

            # 누적 평균 = 손실 / 누적 횟수 의 기울기를 더합니다
            (loss / accumulation_steps).backward()

        optimizer.step()

        # 준비 갱신이 끝나면 최대 메모리 기록과 시작 시각을 다시 잡습니다
        if step == 0:
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            start = time.perf_counter()

    # GPU 계산이 모두 끝난 뒤 시간을 계산합니다
    torch.cuda.synchronize()
    seconds_per_update = (time.perf_counter() - start) / steps
    return {
        "배치 크기": batch_size,
        "시퀀스 길이": seq_len,
        "누적 횟수": accumulation_steps,
        "유효 배치": batch_size * accumulation_steps,
        "최대 할당(GB)": round(torch.cuda.max_memory_allocated() / 1024 ** 3, 2),
        "예약(GB)": round(torch.cuda.max_memory_reserved() / 1024 ** 3, 2),
        "갱신당 시간(초)": round(seconds_per_update, 3),
    }

In [ ]:
# 한 번만 실행합니다(다시 실행하면 지운 이름이 없어 NameError). 앞에서 불러온 모델을 지워 GPU 메모리를 비웁니다
del base_model, base_4bit, qlora_model
gc.collect()
torch.cuda.empty_cache()

## 4. float32 LoRA와 QLoRA의 학습 메모리: load_lora_model, measure_train_step

**목적:** 지난 과제의 float32 LoRA 대신 QLoRA로 학습하면 GPU 메모리가 얼마나 줄어드는지 비교합니다.

**요구사항**
- (1) `load_mode`를 `"float32"`, `"4bit"` 순서로 돌며 `load_lora_model`로 그래디언트 체크포인팅을 끈 모델을 `model`에 불러옵니다.
- (2) 모델마다 교안 01과 같은 `torch.optim.AdamW` 옵티마이저(모델의 파라미터 전체, 학습률 `2e-4`)를 `optimizer`에 만듭니다.
- (3) `measure_train_step`으로 `commerce_token_ids`를 배치 크기 4, 시퀀스 길이 256(누적 없음)으로 측정한 딕셔너리를 **`load_results`**`[load_mode]`에 담습니다.
- (4) 측정이 끝날 때마다 `model`, `optimizer`를 지우고 GPU 메모리를 비웁니다.
- (5) `load_results`의 키를 행(인덱스 `"float32"`, `"4bit"`)으로, 측정 키를 열로 편 `pd.DataFrame` **`load_table`** 을 만듭니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 6-2 float32 LoRA, bf16 LoRA, QLoRA 비교 셀과 같은 방법입니다.
```

</details>

In [ ]:
# [작성] 두 방식으로 갱신을 측정하고 방식 비교 테이블을 만드세요.

# float32 LoRA(지난 과제와 같은 베이스)와 QLoRA(4비트 베이스)
load_results = {}
for load_mode in ["float32", "4bit"]:
    # TODO (1) load_lora_model 에 MODEL_NAME, load_mode 를 넘기고 체크포인팅을 끈 모델을 model 에 담습니다
    ...

    # TODO (2) model.parameters() 와 학습률 2e-4 로 torch.optim.AdamW 를 만들어 optimizer 에 담습니다
    ...

    # 첫 측정에서 빠른 커널 대신 PyTorch 기본 구현으로 계산한다는 안내(causal_conv1d, flash-linear-attention 이 없다)가 나옵니다. 오류가 아닙니다
    # TODO (3) measure_train_step 에 model, optimizer, commerce_token_ids, 배치 크기 4, 시퀀스 길이 256 을 넘긴 결과를 load_results[load_mode] 에 담습니다
    ...

    # TODO (4) model, optimizer 를 지우고 GPU 메모리를 비웁니다
    ...

# TODO (5) pd.DataFrame.from_dict 로 load_results 의 키를 행으로 폅니다(orient 인자)
load_table = ...
display(load_table)

In [ ]:
# [자가채점] 방식 비교 테이블의 인덱스, 열, 측정 설정, 메모리 크기 순서를 확인합니다
assert isinstance(load_results, dict) and list(load_results) == ["float32", "4bit"], f"load_results의 키가 {list(load_results)}입니다. \"float32\", \"4bit\" 순서로 측정하세요."
assert isinstance(load_table, pd.DataFrame) and list(load_table.index) == ["float32", "4bit"], f"load_table의 인덱스가 {list(load_table.index)}입니다. load_results 를 행으로 펴세요."
assert list(load_table.columns) == ['배치 크기', '시퀀스 길이', '누적 횟수', '유효 배치', '최대 할당(GB)', '예약(GB)', '갱신당 시간(초)'], f"load_table의 열이 {list(load_table.columns)}입니다. measure_train_step 이 돌려준 딕셔너리를 그대로 담으세요."
for load_mode in ["float32", "4bit"]:
    row = load_table.loc[load_mode]
    assert row["배치 크기"] == 4 and row["시퀀스 길이"] == 256 and row["누적 횟수"] == 1, f"{load_mode} 행의 측정 설정이 다릅니다. 배치 크기 4, 시퀀스 길이 256, 누적 없이 측정하세요."
    assert 0 < row["최대 할당(GB)"] <= row["예약(GB)"], f"{load_mode} 행의 최대 할당이 0 이하이거나 예약보다 큽니다. measure_train_step 의 반환값을 그대로 담았는지 확인하세요."
    assert row["갱신당 시간(초)"] > 0, f"{load_mode} 행의 갱신당 시간이 0 이하입니다. measure_train_step 의 반환값을 고치지 말고 그대로 담았는지 확인하세요."
assert load_table.loc["4bit", "최대 할당(GB)"] < load_table.loc["float32", "최대 할당(GB)"], "4bit의 최대 할당 메모리가 float32보다 작지 않습니다. load_lora_model 에 load_mode 를 넘겼는지, 측정 사이에 모델을 지웠는지 확인하세요."
print("✅ 통과!")

## 5. 그래디언트 체크포인팅: load_lora_model, measure_train_step

**목적:** 그래디언트 체크포인팅을 켜면 QLoRA 학습 메모리가 얼마나 더 줄어드는지 측정합니다.

**요구사항**
- (1) `load_lora_model`로 그래디언트 체크포인팅을 켠 QLoRA 모델(`load_mode`는 `"4bit"`)을 `model`에 불러옵니다.
- (2) 4번과 같은 AdamW 옵티마이저(학습률 `2e-4`)를 `optimizer`에 만듭니다.
- (3) 키가 `"끔"`, `"켬"`(이 순서)인 딕셔너리 **`checkpointing_results`** 를 만들고, `"끔"`에는 4번의 `load_results["4bit"]`를 그대로, `"켬"`에는 이 모델을 4번과 같은 배치 크기 4, 시퀀스 길이 256으로 `measure_train_step`한 결과를 담습니다.
- (4) `checkpointing_results`의 키를 행(인덱스 `"끔"`, `"켬"`)으로, 측정 키를 열로 편 `pd.DataFrame` **`checkpointing_table`** 을 만듭니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 7-1 그래디언트 체크포인팅 셀과 같은 방법입니다.
```

</details>

In [ ]:
# [작성] 체크포인팅을 켠 QLoRA 를 측정해 끔과 함께 담고, 끔/켬 테이블을 만드세요.

# TODO (1) load_lora_model 로 "4bit" 모델을 체크포인팅을 켜고 불러옵니다
model = ...

# TODO (2) 4번과 같은 torch.optim.AdamW 옵티마이저를 만듭니다
optimizer = ...

# TODO (3) "끔" 은 load_results["4bit"] 를 그대로, "켬" 은 measure_train_step 으로 새로 측정한 결과를 담습니다(배치 크기 4, 시퀀스 길이 256)
checkpointing_results = ...

# 다음 측정을 같은 조건에서 하도록 모델과 옵티마이저를 지웁니다(그대로 씁니다)
del model, optimizer
gc.collect()
torch.cuda.empty_cache()

# TODO (4) 4번과 같이 pd.DataFrame.from_dict 로 키를 행으로 폅니다
checkpointing_table = ...
display(checkpointing_table)

In [ ]:
# [자가채점] 끔/켬 테이블의 인덱스, 열, 끔 행의 값, 메모리 크기 순서를 확인합니다
assert isinstance(checkpointing_results, dict) and list(checkpointing_results) == ["끔", "켬"], f"checkpointing_results의 키가 {list(checkpointing_results)}입니다. \"끔\", \"켬\" 순서입니다."
assert checkpointing_results["끔"] == load_results["4bit"], "끔은 다시 측정하지 말고 load_results[\"4bit\"]를 그대로 담으세요."
on_result = checkpointing_results["켬"]
assert list(on_result) == ['배치 크기', '시퀀스 길이', '누적 횟수', '유효 배치', '최대 할당(GB)', '예약(GB)', '갱신당 시간(초)'], "켬은 measure_train_step 이 돌려준 딕셔너리입니다."
assert on_result["배치 크기"] == 4 and on_result["시퀀스 길이"] == 256, "켬도 4번과 같은 배치 크기 4, 시퀀스 길이 256으로 측정하세요."
assert isinstance(checkpointing_table, pd.DataFrame) and list(checkpointing_table.index) == ["끔", "켬"], f"checkpointing_table의 인덱스가 {list(checkpointing_table.index)}입니다. checkpointing_results 를 행으로 펴세요."
assert list(checkpointing_table.columns) == ['배치 크기', '시퀀스 길이', '누적 횟수', '유효 배치', '최대 할당(GB)', '예약(GB)', '갱신당 시간(초)'], f"checkpointing_table의 열이 {list(checkpointing_table.columns)}입니다."
assert checkpointing_table.loc["켬", "최대 할당(GB)"] < checkpointing_table.loc["끔", "최대 할당(GB)"], "체크포인팅을 켠 최대 할당 메모리가 끔보다 작지 않습니다. load_lora_model 에 gradient_checkpointing=True 를 넘겼는지 확인하세요."
print("✅ 통과!")

## 6. 메모리 예산 안에서 학습 설정 고르기: measure_train_step, SFTConfig

**목적:** 정해진 GPU 메모리 예산 안에서 가장 빨리 학습하는 설정을 골라 학습 설정으로 옮깁니다.

**요구사항**
- (1) 셀에 적어 둔 `commerce_candidates`의 후보 `(batch_size, seq_len, accumulation_steps, gradient_checkpointing)`마다 하한 **`lower_bound_gb`**(4비트 가중치 `weight_4bit_gb` + 배치 크기 * 시퀀스 길이 * `vocab_size` * 4바이트를 `1024 ** 3`으로 나눈 값)를 반올림하지 않고 계산합니다.
- (2) `lower_bound_gb`가 `memory_budget_gb`보다 크면 그 후보 튜플을 그대로 리스트 **`skipped_candidates`** 에 더하고 측정하지 않습니다.
- (3) 나머지 후보는 `load_lora_model`로 그 후보의 체크포인팅 설정을 넣은 QLoRA 모델(`"4bit"`)을 `model`에 새로 불러옵니다.
- (4) 4번과 같은 AdamW 옵티마이저(학습률 `2e-4`)를 `optimizer`에 만듭니다.
- (5) `measure_train_step`으로 `commerce_token_ids`를 그 후보의 배치 크기, 시퀀스 길이, 누적 횟수(`accumulation_steps` 인자)로 측정합니다.
- (6) 측정 딕셔너리에 키 `"체크포인팅"`(그 후보의 `True` 또는 `False`)을 더해 리스트 **`commerce_results`** 에 후보 순서대로 모읍니다.
- (7) `commerce_results`로 행이 측정한 후보 순서이고 열이 측정 키 뒤에 `"체크포인팅"`인 `pd.DataFrame` **`commerce_table`** 을 만듭니다.
- (8) `commerce_results`를 앞부터 보며 `"예약(GB)"`이 `memory_budget_gb` 이하인 후보 가운데 `"갱신당 시간(초)"`이 가장 짧은 딕셔너리를 **`chosen`** 에 담습니다(같으면 앞의 후보, 없으면 `None`). `chosen`이 `None`이면 교안 01 8-2와 같이 아래 `SFTConfig` 줄에서 멈추므로, 그때는 `"예약(GB)"`을 `memory_budget_gb`와 비교했는지부터 확인합니다.
- (9) `trl`의 `SFTConfig`로 **`chosen_args`** 를 만들고 `output_dir`에는 `output_dir / "commerce_qlora_chosen"`의 경로 문자열을 넣습니다(학습은 하지 않습니다).
- (10) `chosen_args`의 `per_device_train_batch_size`, `gradient_accumulation_steps`, `gradient_checkpointing`, `max_length`에 `chosen`의 `"배치 크기"`, `"누적 횟수"`, `"체크포인팅"`, `"시퀀스 길이"`를 차례로 넣습니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 8-2 후보를 측정해 설정 고르기 셀과 같은 방법입니다.
교안과 달리 후보마다 체크포인팅 설정이 달라, 반복문 안에서 후보마다 모델을 새로 불러옵니다.
SFTConfig 는 지난 과제와 같은 방법으로 만들고, 고른 값을 넣을 인자 이름은 교안 01 9절 테이블에 있습니다.
```

</details>

In [ ]:
# [작성] 로짓 하한이 예산을 넘는 후보를 빼고, 나머지 후보를 측정해 후보 테이블을 만드세요.

# 메모리 예산(GB, 1024 ** 3 바이트), 어휘 크기(모델 설정 파일의 vocab_size), 4비트 가중치(GB, 교안 01 계산값)(그대로 씁니다)
memory_budget_gb = 12
vocab_size = 248320
weight_4bit_gb = 0.71

# 유효 배치 16인 후보: (배치 크기, 시퀀스 길이, 누적 횟수, 체크포인팅)(그대로 씁니다)
commerce_candidates = [
    (16, 1024, 1, False),                 # 모든 예시를 1024토큰으로 채웠다고 본 경우
    (16, commerce_max_length, 1, False),  # 가장 긴 예시에 맞춘 길이
    (4, commerce_max_length, 4, True),    # 배치를 줄이고 누적으로 유효 배치 유지, 체크포인팅 켬
]

commerce_results = []
skipped_candidates = []
for batch_size, seq_len, accumulation_steps, gradient_checkpointing in commerce_candidates:
    # TODO (1) 하한(GB): weight_4bit_gb 에 배치 크기 * 시퀀스 길이 * vocab_size * 4 를 1024 ** 3 으로 나눈 값을 더합니다
    lower_bound_gb = ...

    # TODO (2) 하한이 memory_budget_gb 보다 크면 후보 네 값의 튜플을 skipped_candidates 에 더합니다
    if ...:
        ...
    else:
        # TODO (3) load_lora_model 로 "4bit" 모델을 이 후보의 gradient_checkpointing 값으로 불러와 model 에 담습니다
        ...

        # TODO (4) 4번과 같은 torch.optim.AdamW 를 optimizer 에 만듭니다
        ...

        # TODO (5) measure_train_step 에 이 후보의 batch_size, seq_len 과 accumulation_steps 인자를 넘겨 측정합니다
        ...

        # TODO (6) 측정 딕셔너리에 "체크포인팅" 키로 gradient_checkpointing 을 더하고 commerce_results 에 더합니다
        ...

        # 다음 후보를 같은 조건에서 측정하도록 모델과 옵티마이저를 지웁니다(그대로 씁니다)
        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()

# TODO (7) 딕셔너리 리스트 commerce_results 를 pd.DataFrame 으로 만듭니다
commerce_table = ...
display(commerce_table)

In [ ]:
# [자가채점] 계산으로 뺀 후보, 후보 테이블의 열과 순서, 유효 배치, 메모리 크기 순서를 확인합니다
expected_candidates = [(16, 1024, 1, False), (16, commerce_max_length, 1, False), (4, commerce_max_length, 4, True)]
assert commerce_candidates == expected_candidates, "셀에 적어 둔 후보 세 개를 바꾸지 마세요."
assert memory_budget_gb == 12 and vocab_size == 248320 and weight_4bit_gb == 0.71, "memory_budget_gb, vocab_size, weight_4bit_gb 는 셀에 적어 둔 값을 그대로 씁니다."
last_batch, last_len = commerce_candidates[-1][0], commerce_candidates[-1][1]
expected_lower_gb = weight_4bit_gb + last_batch * last_len * vocab_size * 4 / 1024 ** 3
assert abs(lower_bound_gb - expected_lower_gb) < 0.01, f"마지막 후보의 lower_bound_gb가 {lower_bound_gb:.2f}GB입니다(기대 {expected_lower_gb:.2f}GB). weight_4bit_gb 에 배치 크기 * 시퀀스 길이 * vocab_size * 4 / 1024 ** 3 을 더했는지 확인하세요."
assert skipped_candidates == [commerce_candidates[0]], f"skipped_candidates가 {skipped_candidates}입니다. 하한(weight_4bit_gb + 배치 크기 * 시퀀스 길이 * vocab_size * 4 / 1024 ** 3)이 memory_budget_gb 보다 큰 후보 튜플만 더하세요."
assert isinstance(commerce_results, list) and len(commerce_results) == 2, f"commerce_results가 {len(commerce_results)}개입니다. 측정한 후보 두 개의 딕셔너리를 모으세요."
assert isinstance(commerce_table, pd.DataFrame) and list(commerce_table.columns) == ['배치 크기', '시퀀스 길이', '누적 횟수', '유효 배치', '최대 할당(GB)', '예약(GB)', '갱신당 시간(초)', '체크포인팅'], f"commerce_table의 열이 {list(commerce_table.columns)}입니다. 측정 키 뒤에 \"체크포인팅\"을 더한 딕셔너리로 만드세요."
measured = []
for row in commerce_table.to_dict("records"):
    measured.append((row["배치 크기"], row["시퀀스 길이"], row["누적 횟수"], row["체크포인팅"]))
    assert row["유효 배치"] == 16, f"유효 배치가 {row['유효 배치']}입니다. measure_train_step 의 accumulation_steps 인자로 후보의 누적 횟수를 넘겨 배치 크기 * 누적 횟수가 16이 되게 하세요."
assert measured == commerce_candidates[1:], f"측정한 (배치 크기, 시퀀스 길이, 누적 횟수, 체크포인팅)이 {measured}입니다. 후보 순서대로 네 값을 넘기고 \"체크포인팅\" 키를 더하세요."
allocated = commerce_table["최대 할당(GB)"]
assert allocated.iloc[1] < allocated.iloc[0], "배치 크기 4(체크포인팅 켬) 행의 최대 할당 메모리가 배치 크기 16 행보다 작지 않습니다. 후보마다 그 체크포인팅 설정으로 모델을 새로 불러왔는지, 배치 크기와 accumulation_steps 를 후보 값으로 넘겼는지 확인하세요."
print("✅ 통과!")

In [ ]:
# [작성] 예산 안에서 갱신당 시간이 가장 짧은 후보를 고르고, 그 설정을 SFTConfig 로 옮기세요.
from trl import SFTConfig

chosen = None
for result in commerce_results:
    # TODO (8) "예약(GB)" 이 memory_budget_gb 이하이고, chosen 이 None 이거나 "갱신당 시간(초)" 가 chosen 보다 짧을 때만 chosen 을 바꿉니다
    ...
display(chosen)

chosen_args = SFTConfig(
    # TODO (9) output_dir 에 output_dir / "commerce_qlora_chosen" 을 str 로 바꿔 넣습니다
    # TODO (10) chosen 의 배치 크기, 누적 횟수, 체크포인팅, 시퀀스 길이를 네 인자에 한 줄에 하나씩 넣습니다
    ...
)

In [ ]:
# [자가채점] 고르는 규칙의 불변식과 옮긴 학습 설정을 확인합니다(어느 후보가 뽑혀야 하는지는 보지 않습니다)
assert chosen is not None, "예산 안의 후보가 없습니다. 예약(GB)을 memory_budget_gb 와 비교했는지 확인하세요."
assert chosen in commerce_results, "chosen은 commerce_results 의 측정 딕셔너리 하나입니다."
assert chosen["예약(GB)"] <= memory_budget_gb, f"고른 후보의 예약 메모리가 예산 {memory_budget_gb}GB를 넘습니다. 예약(GB)을 예산과 비교하세요."
chosen_index = commerce_results.index(chosen)
for index, result in enumerate(commerce_results):
    if result["예약(GB)"] <= memory_budget_gb:
        assert result["갱신당 시간(초)"] >= chosen["갱신당 시간(초)"], f"예산 안의 {index}번 후보가 더 빠릅니다. 갱신당 시간이 가장 짧은 후보를 고르세요."
        if index < chosen_index:
            assert result["갱신당 시간(초)"] > chosen["갱신당 시간(초)"], f"앞의 {index}번 후보의 갱신당 시간이 같습니다. 같으면 앞의 후보를 고릅니다."
assert isinstance(chosen_args, SFTConfig), "chosen_args는 trl의 SFTConfig입니다. `...`를 채웠는지 확인하세요."
assert chosen_args.per_device_train_batch_size == chosen["배치 크기"] and chosen_args.gradient_accumulation_steps == chosen["누적 횟수"], "chosen_args의 per_device_train_batch_size, gradient_accumulation_steps 에 chosen 의 배치 크기, 누적 횟수를 넣으세요."
assert chosen_args.gradient_checkpointing == chosen["체크포인팅"], f"chosen_args의 gradient_checkpointing이 {chosen_args.gradient_checkpointing}입니다. chosen 의 체크포인팅 설정을 넣으세요(SFTConfig 기본값은 True)."
assert chosen_args.max_length == chosen["시퀀스 길이"], "chosen_args의 max_length 에 chosen 의 시퀀스 길이를 넣으세요."
assert chosen_args.output_dir.endswith("commerce_qlora_chosen"), f"output_dir가 {chosen_args.output_dir}입니다. output 폴더 아래 commerce_qlora_chosen 폴더의 경로를 문자열로 넣으세요."
print("✅ 통과!")